# D1 · Piloto de proyección sobre DINOv2 congelado
Objetivo/pregunta: comprobar si una métrica lineal mejora validación frente a C1 y medir coste local. Hipótesis: el ajuste ayuda, pero puede degradar. Intervención: matriz identidad 1024×1024 entrenada, encoder congelado. Control: C1 en época cero. Muestra: 1.040 train locales verificados contra v2 y 273 val; **piloto reducido, no estudio completo de 193.276 fotos**. Semillas 45–47, hasta 30×100 pasos, P8×K4, lr 1e-4, margen .2, paciencia 7. Selección por mAP macro de validación, época cero elegible. No se evalúa test en este piloto. Galería cerrada, misma especie, leave-one-out por fotografía; Rank-1 por identidad, AP por fotografía. No confirma generalización ni separación por sesión.
Métodos externos: [DINOv2](https://arxiv.org/abs/2304.07193), [batch-hard](https://arxiv.org/abs/1703.07737). La proyección identidad y este protocolo son decisiones del estudio, no métodos nuevos acreditados.
## Fase 1. Preparación y verificación de datos
Control previo: 236 fotos train del caché antiguo no están en el corpus fijado y se excluyen de la selección del piloto. Se conserva el intento fallido y el inventario; todas las fotos de validación coinciden.

In [1]:
import sys
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Python 3.11–3.13 required")
import os, json, csv, time, hashlib
from pathlib import Path
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
from dotenv import load_dotenv
load_dotenv('.env')
import numpy as np
import torch
import mlflow
from huggingface_hub import snapshot_download
from pettrace.data.refined_sources import RELEASES
from pettrace.data.verification import sha256
from pettrace.tracking import PRODUCTION_URI, source_identity
from pettrace.training.dino_metric import Metric, fit
assert torch.backends.mps.is_available()
torch.set_num_threads(4)
LOG_TO_MLFLOW = True
mlflow.set_tracking_uri(PRODUCTION_URI)
mlflow.set_registry_uri(PRODUCTION_URI)
REPORT = Path('reports/d1-dino-pilot/20261118-local-v1')
REPORT.mkdir(parents=True, exist_ok=True)
client = mlflow.MlflowClient()
# Create once; reruns recover publication without training again.
receipt_path = REPORT/'tracking.json'
if receipt_path.exists():
    run_id = json.loads(receipt_path.read_text())['run_id']
else:
    run_id = client.create_run('3', tags={'mlflow.runName':'D1 DINO metric pilot, train/val only', 'experiment':'D1', 'purpose':'reduced_training_pilot'}).info.run_id
    receipt_path.write_text(json.dumps({'run_id':run_id}))
manifest = next(Path('reports/.pettrace-training').rglob('versions/v1/manifest.csv'))
local = list(csv.DictReader(manifest.open()))
lookup, provenance = {}, {}
for name, spec in RELEASES.items():
    root = Path(snapshot_download(spec['repo_id'], repo_type='dataset', revision=spec['v2_commit'], allow_patterns=['versions/v1/dataset.json','versions/v1/metadata.csv']))
    evidence = json.loads((root/'versions/v1/dataset.json').read_text())
    assert sha256(root/'versions/v1/metadata.csv') == evidence['file_sha256']['versions/v1/metadata.csv']
    provenance[name] = dict(commit=spec['v2_commit'], metadata_sha256=sha256(root/'versions/v1/metadata.csv'))
    for row in csv.DictReader((root/'versions/v1/metadata.csv').open()):
        assert row['sha256'] not in lookup
        lookup[row['sha256']] = row
excluded=[r['sha256'] for r in local if r['split']=='train' and r['sha256'] not in lookup]
assert len(excluded)==236
(REPORT/'excluded-local-cache.json').write_text(json.dumps({'reason':'Not in immutable current prepared corpus','sha256':excluded},indent=2))
rows=[]
for row in local:
    if row['split'] not in ('train','val'): continue
    if row['sha256'] not in lookup:
        assert row['split']=='train'
        continue
    official=lookup[row['sha256']]
    assert official['split']==row['split'] and official['species']==row['species']
    path=manifest.parent/row['path']
    assert sha256(path)==row['sha256']
    rows.append(dict(path=str(path),animal=official['identity_id'],species=official['species'],split=official['split'],sha256=row['sha256']))
assert sum(r['split']=='train' for r in rows)==1040
assert sum(r['split']=='val' for r in rows)==273
assert not {r['animal'] for r in rows if r['split']=='train'} & {r['animal'] for r in rows if r['split']=='val'}
(REPORT/'rows.json').write_text(json.dumps(rows,indent=2))
print('Verified train=1040 val=273; MLflow run',run_id,flush=True)


2026/11/18 10:48:14 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Verified train=1040 val=273; MLflow run c28ea6cec2bc41a4a12435c5f7e08493


## Fase 2. Representación y entrenamiento
Encoder y procesador iguales a C1. Recorte E2 online sobre bytes verificados; sin aumentos. Caché independiente con procedencia y checksum, recuperable entre ejecuciones. Entrenamiento de la proyección en CPU; extracción MPS.

In [2]:
from transformers import AutoImageProcessor, AutoModel
from PIL import Image, ImageOps
from pettrace.evaluation.core import load_model
revision='47b73eefe95e8d44ec3623f8890bd894b6ea2d6c'
model_dir=Path(snapshot_download('facebook/dinov2-large',revision=revision,allow_patterns=['model.safetensors','config.json','preprocessor_config.json']))
processor=AutoImageProcessor.from_pretrained(str(model_dir),use_fast=False)
backbone=AutoModel.from_pretrained(str(model_dir)).eval().requires_grad_(False).to('mps')
base=load_model('models:/pettrace-embedder/2')
assert base.golden_ok and base.pipeline.detector is not None
identity=dict(model_revision=revision, model_hashes={f:sha256(model_dir/f) for f in ('model.safetensors','config.json','preprocessor_config.json')},detector_hashes=base.sha256,processor=processor.to_dict(),rows_sha256=sha256(REPORT/'rows.json'), source=source_identity(),provenance=provenance,training_module_sha256=sha256('src/pettrace/training/dino_metric.py'),notebook_sha256=sha256('notebooks/d1_dinov2_pilot.ipynb'))
(REPORT/'design.json').write_text(json.dumps(identity,indent=2))
client.log_artifact(run_id,str(REPORT/'design.json'),'report')
cache=REPORT/'features'
cache.mkdir(exist_ok=True)
cache_key=hashlib.sha256(json.dumps({k:identity[k] for k in ('model_hashes','detector_hashes','processor','rows_sha256')},sort_keys=True).encode()).hexdigest()
start=time.monotonic()
parts=[]
for offset in range(0,len(rows),8):
    target=cache/f'{cache_key}-{offset}.npy'
    checksum=target.with_suffix('.sha256')
    if target.exists() and checksum.exists():
        assert sha256(target)==checksum.read_text()
        values=np.load(target,allow_pickle=False)
    else:
        images=[]
        for row in rows[offset:offset+8]:
            with Image.open(row['path']) as image:
                images.append(base.pipeline.detector.crop(ImageOps.exif_transpose(image).convert('RGB')))
        pixels=processor(images=images,return_tensors='pt')['pixel_values'].to('mps')
        with torch.inference_mode():
            values=torch.nn.functional.normalize(backbone(pixel_values=pixels).last_hidden_state[:,0],dim=-1).cpu().numpy()
        np.save(target,values)
        checksum.write_text(sha256(target))
    assert values.shape==(min(8,len(rows)-offset),1024) and np.isfinite(values).all()
    parts.append(values)
    if offset%200==0:print('Extracted',offset+len(values),'/',len(rows),flush=True)
features=np.concatenate(parts)
elapsed=time.monotonic()-start
if not (REPORT/'extraction.json').exists():
    (REPORT/'extraction.json').write_text(json.dumps(dict(seconds=elapsed,photos=len(rows),backend='MPS with CPU fallback',batch=8),indent=2))
assert np.allclose(Metric()(torch.tensor(features[:8])).detach().numpy(),features[:8],atol=1e-6)
del backbone
results=[]
for seed in (45,46,47):
    folder=REPORT/f'seed-{seed}'
    if (folder/'result.json').exists():
        result=json.loads((folder/'result.json').read_text())
    else:
        _,result=fit(features,rows,folder,seed)
    results.append(result)
    client.log_artifacts(run_id,str(folder),f'report/seed-{seed}')


Extracted 8 / 1313


Extracted 208 / 1313


Extracted 408 / 1313


Extracted 608 / 1313


Extracted 808 / 1313


Extracted 1008 / 1313


Extracted 1208 / 1313


## Fase 3. Evaluación y análisis
Resultados de validación exclusivamente, con tres semillas y C1. Publicación obligatoria de evidencia y notebook; registro/promoción de modelo no aplican a este piloto técnico. La selección de la familia y los tests pertenecen al estudio completo. Las métricas del piloto no sustituyen resultados de test ni prueban mejora publicable.

In [3]:
initial=results[0]['initial']
summary=dict(control_validation=initial,seeds=[dict(seed=r['seed'],best_epoch=r['best_epoch'],validation=r['best']) for r in results],mean_validation={k:float(np.mean([r['best'][k] for r in results])) for k in ('mAP','rank1')},sample=dict(train=1040,val=273,test_evaluated=False),limitations=['Reduced training sample','Validation only','No capture-session split','No independent confirmation'],published=False)
(REPORT/'summary.json').write_text(json.dumps(summary,indent=2))
for r in results:
    for k,v in r['best'].items():client.log_metric(run_id,f'val.seed{r["seed"]}.{k}',v)
for k,v in initial.items():client.log_metric(run_id,f'val.control.{k}',v)
for name in ('rows.json','design.json','extraction.json','summary.json','excluded-local-cache.json','preflight-mismatch.ipynb','interrupted-publication.ipynb','initial-design.json'):
    client.log_artifact(run_id,str(REPORT/name),'report')
client.log_artifact(run_id,'src/pettrace/training/dino_metric.py','source')
client.set_terminated(run_id,'FINISHED')
assert client.get_run(run_id).info.status=='FINISHED'
download=Path(client.download_artifacts(run_id,'report/summary.json',str(REPORT/'verify')))
assert sha256(download)==sha256(REPORT/'summary.json')
(REPORT/'publication.json').write_text(json.dumps(dict(run_id=run_id,published=True,verified_summary_sha256=sha256(download)),indent=2))
client.log_artifact(run_id,str(REPORT/'publication.json'),'report')
print(json.dumps(summary,indent=2))
print('Published and verified:',run_id)


🏃 View run D1 DINO metric pilot, train/val only at: https://ml.pettrace.app/#/experiments/3/runs/c28ea6cec2bc41a4a12435c5f7e08493
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


{
  "control_validation": {
    "mAP": 0.8387034377752959,
    "rank1": 0.8946737311767987
  },
  "seeds": [
    {
      "seed": 45,
      "best_epoch": 3,
      "validation": {
        "mAP": 0.8672715551165435,
        "rank1": 0.9175125488008924
      }
    },
    {
      "seed": 46,
      "best_epoch": 11,
      "validation": {
        "mAP": 0.869427010220482,
        "rank1": 0.9205800334634691
      }
    },
    {
      "seed": 47,
      "best_epoch": 18,
      "validation": {
        "mAP": 0.8709037135531653,
        "rank1": 0.9236475181260457
      }
    }
  ],
  "mean_validation": {
    "mAP": 0.8692007596300636,
    "rank1": 0.9205800334634691
  },
  "sample": {
    "train": 1040,
    "val": 273,
    "test_evaluated": false
  },
  "limitations": [
    "Reduced training sample",
    "Validation only",
    "No capture-session split",
    "No independent confirmation"
  ],
  "published": false
}
Published and verified: c28ea6cec2bc41a4a12435c5f7e08493


**Resultado publicado y verificado:** tres semillas; mAP macro de validación medio 0,869201 frente a 0,838703 del control; Rank-1 macro medio 92,058 % frente a 89,467 %. Solo piloto reducido de 1.040 train / 273 val, sin test. El campo `published: false` en el resumen refleja su guardado previo a la publicación; el recibo `publication.json` posterior acredita el éxito. [MLflow](https://ml.pettrace.app/#/experiments/3/runs/c28ea6cec2bc41a4a12435c5f7e08493). [Informe](../docs/d1_dinov2_pilot_results_20261118.md).